In [29]:
import os
import pandas as pd, math
import numpy as np
from pyhive import presto
from datetime import datetime, date, timedelta
import warnings

# Ignore a specific warning by its type
warnings.filterwarnings("ignore")

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

presto_port = '80'
username = 'airflow-user'
# username = 'nihal.r@rapido.bike'

connection1 = presto.connect('bi-presto.serving.data.production.internal', presto_port, username)
connection2 = presto.connect('bi-trino-2.serving.data.production.internal', presto_port, username)
connection3= presto.connect('processing-2.processing.data.production.internal',presto_port,username)
connection4= presto.connect('processing.processing.data.production.internal',presto_port,username)


In [61]:
def consideration_query(run_date,city_name,base_week):
    
    rr_data_query = f"""
  
        with dormant_customer as
    (
    select
    customer_id as customer,
    date_diff('day',taxi_lifetime_first_ride_date,date(run_date)) rapido_first_ride,
    ceil(cast(date_diff('day',taxi_lifetime_first_ride_date,date(run_date)) as real)/10)*10 rapido_age_seg_fr,
    taxi_lifetime_last_ride_date,
    taxi_lifetime_first_ride_date,
    taxi_lifetime_stage,
    previous_taxi_lifetime_stage,
    taxi_lifetime_rides,
    taxi_recency_segment,
    taxi_rr_active_days_last_21_days as run_taxi_rr_active_days_last_21_days,
    taxi_lifetime_last_ride_city city_name,
    fe_mean_intent_type,
    fe_intent_trend_type,
    fe_potential_per_type,
    fe_regularity_type,
    fe_recency_type,
    fe_volatility_type,
    gender_tag,
    taxi_income_segment,
    ps_tag_link,
    ps_tag_auto,
    taxi_service_affinity,
    taxi_distance_preference,
    taxi_time_of_day_preference,
    taxi_day_of_week_preference
    from datasets.iallocator_customer_segments
    where date(run_date) = date('{base_week}')
    -- and taxi_lifetime_last_ride_date = date('2023-11-01')
    and taxi_recency_segment in ('RECENT')
    and taxi_lifetime_rides > 4
    and taxi_lifetime_last_ride_city in ('{city_name}')
    )
    ,
    base as 
(
select a.*,
case when coalesce(b.rr,0) <1 then 0 else 1 end as rr_retention_flag,
case when coalesce(b.fe,0) <1 then 0 else 1 end as fe_retention_flag,
case when coalesce(b.ao,0) <1 then 0 else 1 end as ao_retention_flag,
case when coalesce(b.nr,0) <1 then 0 else 1 end as net_retention_flag,

case when coalesce(b.nr_post_w01,0) <1 then 0 else 1 end as nr_flag_w01,
case when coalesce(b.nr_post_w02,0) <1 then 0 else 1 end as nr_flag_w02,
case when coalesce(b.nr_post_w03,0) <1 then 0 else 1 end as nr_flag_w03,
case when coalesce(b.nr_post_w04,0) <1 then 0 else 1 end as nr_flag_w04,
case when coalesce(b.nr_post_w05,0) <1 then 0 else 1 end as nr_flag_w05,
case when coalesce(b.nr_post_w06,0) <1 then 0 else 1 end as nr_flag_w06,
case when coalesce(b.nr_post_w07,0) <1 then 0 else 1 end as nr_flag_w07,
case when coalesce(b.nr_post_w08,0) <1 then 0 else 1 end as nr_flag_w08,
case when coalesce(b.nr_post_w09,0) <1 then 0 else 1 end as nr_flag_w09,
case when coalesce(b.nr_post_w10,0) <1 then 0 else 1 end as nr_flag_w10,

case when coalesce(b.rr_post_w01,0) <1 then 0 else 1 end as rr_flag_w01,
case when coalesce(b.rr_post_w02,0) <1 then 0 else 1 end as rr_flag_w02,
case when coalesce(b.rr_post_w03,0) <1 then 0 else 1 end as rr_flag_w03,
case when coalesce(b.rr_post_w04,0) <1 then 0 else 1 end as rr_flag_w04,
case when coalesce(b.rr_post_w05,0) <1 then 0 else 1 end as rr_flag_w05,
case when coalesce(b.rr_post_w06,0) <1 then 0 else 1 end as rr_flag_w06,
case when coalesce(b.rr_post_w07,0) <1 then 0 else 1 end as rr_flag_w07,
case when coalesce(b.rr_post_w08,0) <1 then 0 else 1 end as rr_flag_w08,
case when coalesce(b.rr_post_w09,0) <1 then 0 else 1 end as rr_flag_w09,
case when coalesce(b.rr_post_w10,0) <1 then 0 else 1 end as rr_flag_w10,

case when coalesce(b.fe_post_w01,0) <1 then 0 else 1 end as fe_flag_w01,
case when coalesce(b.fe_post_w02,0) <1 then 0 else 1 end as fe_flag_w02,
case when coalesce(b.fe_post_w03,0) <1 then 0 else 1 end as fe_flag_w03,
case when coalesce(b.fe_post_w04,0) <1 then 0 else 1 end as fe_flag_w04,
case when coalesce(b.fe_post_w05,0) <1 then 0 else 1 end as fe_flag_w05,
case when coalesce(b.fe_post_w06,0) <1 then 0 else 1 end as fe_flag_w06,
case when coalesce(b.fe_post_w07,0) <1 then 0 else 1 end as fe_flag_w07,
case when coalesce(b.fe_post_w08,0) <1 then 0 else 1 end as fe_flag_w08,
case when coalesce(b.fe_post_w09,0) <1 then 0 else 1 end as fe_flag_w09,
case when coalesce(b.fe_post_w10,0) <1 then 0 else 1 end as fe_flag_w10


from dormant_customer a
left join
(
select customerid,
-- sum(ao_sessions_unique_daily) ao,
-- sum(fe_sessions_unique_daily) fe,
-- sum(rr_sessions_unique_daily) rr,
-- sum(net_rides_daily) nr,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then ao_sessions_unique_daily end) as ao,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then net_rides_daily end) as nr,

sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '70' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w10,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '63' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w09,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '56' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w08,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '49' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w07,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '42' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w06,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '35' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w05,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '28' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w04,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w03,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '14' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w02,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '7' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w01,

sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '70' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w10,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '63' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w09,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '56' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w08,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '49' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w07,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '42' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w06,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '35' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w05,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '28' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w04,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w03,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '14' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w02,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '7' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w01,


sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '70' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w10,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '63' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w09,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '56' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w08,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '49' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w07,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '42' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w06,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '35' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w05,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '28' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w04,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w03,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '14' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w02,
sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '7' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w01

from datasets.customer_rf_daily_kpi 
where date(day)>date('{base_week}') and date(day)<=date('{base_week}')+interval '70' day
and service_name in ('Link','Auto')
and city in ('{city_name}')
group by 1
)b
on a.customer=b.customerid

)
    ,
rr_data as(
select fe.*
    from
    (select aa.*, bb.rr_LT, bb.rapido_age,bb.rapido_age_seg
    from
    (select customerid,
    sum(case when day > date_format(date('{run_date}') + interval '-92' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-85' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w13,
    sum(case when day > date_format(date('{run_date}') + interval '-85' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-78' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w12,
    sum(case when day > date_format(date('{run_date}') + interval '-78' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-71' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w11,
    sum(case when day > date_format(date('{run_date}') + interval '-71' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-64' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w10,
    sum(case when day > date_format(date('{run_date}') + interval '-64' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-57' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w09,
    sum(case when day > date_format(date('{run_date}') + interval '-57' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-50' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w08,
    sum(case when day > date_format(date('{run_date}') + interval '-50' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-43' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w07,
    sum(case when day > date_format(date('{run_date}') + interval '-43' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-36' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w06,
    sum(case when day > date_format(date('{run_date}') + interval '-36' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-29' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w05,
    sum(case when day > date_format(date('{run_date}') + interval '-29' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-22' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w04,
    sum(case when day > date_format(date('{run_date}') + interval '-22' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-15' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w03,
    sum(case when day > date_format(date('{run_date}') + interval '-15' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-08' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w02,
    sum(case when day > date_format(date('{run_date}') + interval '-08' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-01' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w01,
    sum(case when day > date_format(date('{run_date}') + interval '-92' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-01' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_91days,
    sum(case when day > date_format(date('{run_date}') + interval '-92' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-01' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_91days,
    count(distinct case when day > date_format(date('{run_date}') + interval '-92' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-85' day, '%Y-%m-%d')  then day end) as active_rr_days_w13,
    count(distinct case when day > date_format(date('{run_date}') + interval '-85' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-78' day, '%Y-%m-%d')  then day end) as active_rr_days_w12,
    count(distinct case when day > date_format(date('{run_date}') + interval '-78' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-71' day, '%Y-%m-%d')  then day end) as active_rr_days_w11,
    count(distinct case when day > date_format(date('{run_date}') + interval '-71' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-64' day, '%Y-%m-%d')  then day end) as active_rr_days_w10,
    count(distinct case when day > date_format(date('{run_date}') + interval '-64' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-57' day, '%Y-%m-%d')  then day end) as active_rr_days_w09,
    count(distinct case when day > date_format(date('{run_date}') + interval '-57' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-50' day, '%Y-%m-%d')  then day end) as active_rr_days_w08,
    count(distinct case when day > date_format(date('{run_date}') + interval '-50' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-43' day, '%Y-%m-%d')  then day end) as active_rr_days_w07,
    count(distinct case when day > date_format(date('{run_date}') + interval '-43' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-36' day, '%Y-%m-%d')  then day end) as active_rr_days_w06,
    count(distinct case when day > date_format(date('{run_date}') + interval '-36' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-29' day, '%Y-%m-%d')  then day end) as active_rr_days_w05,
    count(distinct case when day > date_format(date('{run_date}') + interval '-29' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-22' day, '%Y-%m-%d')  then day end) as active_rr_days_w04,
    count(distinct case when day > date_format(date('{run_date}') + interval '-22' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-15' day, '%Y-%m-%d')  then day end) as active_rr_days_w03,
    count(distinct case when day > date_format(date('{run_date}') + interval '-15' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-08' day, '%Y-%m-%d')  then day end) as active_rr_days_w02,
    count(distinct case when day > date_format(date('{run_date}') + interval '-08' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-01' day, '%Y-%m-%d')  then day end) as active_rr_days_w01,
    day((date('{run_date}') + interval '-01' day) - date(max(day))) as days_since_last_rr
      from 
        (select customerid,day, max(fe_sessions_unique_daily) as fe_sessions_unique_daily,  sum(rr_sessions_unique_daily) as rr_sessions_unique_daily
        from 
        datasets.customer_rf_daily_kpi a
        join
        base b
        on a.customerid=b.customer
        and 1=1 
        and city = '{city_name}'
        and rr_sessions_unique_daily >= 1 
        and day >= date_format(date('{run_date}') + interval '-91' day , '%Y-%m-%d')
        and day <= date_format(date('{run_date}') + interval '-1' day , '%Y-%m-%d')
        and service_name in ('Link','Auto')
        group by customerid, day
        )
      group by 
        customerid) aa 
    left join 
    (select customerid, sum(rr_sessions_unique_daily) as rr_LT , day((date('{run_date}') + interval '-01' day) - date(min(day))) as rapido_age,
    ceil(cast(day((date('{run_date}') + interval '-01' day) - date(min(day))) as real)/10)*10 rapido_age_seg
            from 
        datasets.customer_rf_daily_kpi a
        join
        base b
        on a.customerid=b.customer
        and rr_sessions_unique_daily >= 1 
        and city = '{city_name}'
        group by 1
        ) bb
        on aa.customerid = bb.customerid) fe 
  )
    select a.*,b.*,'{run_date}' as run_date
    from 
    base a
    left join
    rr_data b
    on a.customer=b.customerid


    """

    try:
        data = pd.read_sql(rr_data_query, connection3)
    except:
        data = pd.read_sql(rr_data_query, connection4)
        
    print(data.shape)   
    columns_to_replace = ['active_rr_days_w13', 'active_rr_days_w12', 'active_rr_days_w11',
                       'active_rr_days_w10', 'active_rr_days_w09', 'active_rr_days_w08',
                       'active_rr_days_w07', 'active_rr_days_w06', 'active_rr_days_w05',
                       'active_rr_days_w04', 'active_rr_days_w03', 'active_rr_days_w02',
                       'active_rr_days_w01']

    data[columns_to_replace] = data[columns_to_replace].mask(data[columns_to_replace] == 0, pd.NA)
    
    data['rr_norm_w13']=data.rr_w13/data.active_rr_days_w13

    data['rr_norm_w12']=data.rr_w12/data.active_rr_days_w12

    data['rr_norm_w11']=data.rr_w11/data.active_rr_days_w11

    data['rr_norm_w10']=data.rr_w10/data.active_rr_days_w10

    data['rr_norm_w09']=data.rr_w09/data.active_rr_days_w09

    data['rr_norm_w08']=data.rr_w08/data.active_rr_days_w08

    data['rr_norm_w07']=data.rr_w07/data.active_rr_days_w07

    data['rr_norm_w06']=data.rr_w06/data.active_rr_days_w06

    data['rr_norm_w05']=data.rr_w05/data.active_rr_days_w05

    data['rr_norm_w04']=data.rr_w04/data.active_rr_days_w04

    data['rr_norm_w03']=data.rr_w03/data.active_rr_days_w03

    data['rr_norm_w02']=data.rr_w02/data.active_rr_days_w02

    data['rr_norm_w01']=data.rr_w01/data.active_rr_days_w01

    raw_customer_data=data[data['customerid']!=0]
    back_up = raw_customer_data.copy()
    fe_data2 = raw_customer_data.fillna(0)
    fe_data=fe_data2[fe_data2['customerid']!=0]
    
    def regularity_decider(x,active_mean):
        if x<=0.3:
            return '05_rare_need'
        elif x>0.3 and x<=0.5:
            return '04_monthly'
        elif x>0.5 and x<=0.75:
            return '03_bi_weekly'
        else:
            if active_mean>2.6:
                return '01_daily'
            else:
                return '02_weekly'
    #         return 'weekly'

    def regularity_decider_v2(x,active_mean):
        if x<=0.2:
            return '05_rare_need'
        elif x>0.2 and x<=0.4:
            return '04_monthly'
        elif x>0.4 and x<=0.689:
            return '03_bi_weekly'
        else:
            if active_mean>2.6:
                return '01_daily'
            else:
                return '02_weekly'

    def intent_decider(x):
        if x<=2:
            return '03_Low'
        elif x>2 and x<=5:
            return '02_Medium'
        else:
            return '01_High'
    


    def consideration_signal_creator(x,y,active_mean):
        if y<=1.5:
            if x<=0.3:
                return 'rarely_1_rr'
            elif x>0.3 and x<=0.5:
                return 'once_in_4_weeks_1_rr'
            elif x>0.5 and x<=0.75:
                return 'once_in_2_weeks_1_rr'
            else:
                return 'weekly_1_rr'
        else:
            if x<=0.3:
                return 'rarely_2+_rr'
            elif x>0.3 and x<=0.5:
                return 'once_in_4_weeks_2+_rr'
            elif x>0.5 and x<=0.75:
                return 'once_in_2_weeks_2+_rr'
            else:
                return 'weekly_2+_rr'
    #             if active_mean>2.6:
    #                 return 'daily'
    #             else:
    #                 return 'weekly'



    def correction_factor(rr_weight,rapido_age):
        if rapido_age >= 91:
            multiplier = 1
            week_inactive = 0

        elif rapido_age == 0:
            week_inactive = (91-rapido_age)//7 - 1
            multiplier = (rr_weight.sum()/(rr_weight.sum()- (week_inactive*(week_inactive+1)/2)))

        elif rapido_age%7 == 0:
            week_inactive = (91-rapido_age)//7 - 1
            multiplier = (rr_weight.sum()/(rr_weight.sum()- (week_inactive*(week_inactive+1)/2)))        
        else:
            week_inactive = (91-rapido_age)//7
            multiplier = (rr_weight.sum()/(rr_weight.sum()- (week_inactive*(week_inactive+1)/2))) 
        return(multiplier,week_inactive,(rr_weight.sum()- (week_inactive*(week_inactive+1)/2)) , ((rr_weight*rr_weight).sum()- (week_inactive*(week_inactive+1)*(2*week_inactive+1)/6)))

    def recency_type(days_since_last_fe):
        if days_since_last_fe <= 6: 
            return('Recent')
        elif days_since_last_fe <= 20:
            return('Stationary')
        else:
            return('Dormant')

    def intent_trend_type(intent_trend):
        if intent_trend <= -0.1:
            return('03_Declining')
        elif intent_trend <= 0.1:
            return('02_Stable')
        else:
            return('01_Increasing')

#     def custom_func(mean_intent,rr_trend):
#         if mean_intent=='High':
#             return rr_trend
#         elif mean_intent=='Low' and rr_trend=='01_Increasing':
#             return '01_Increasing'
#         else:
#             return '02_Stable'

    def create_L3_signals(fe_data,back_up): 
        
        rr_weight = np.array([1,2,3,4,5,6,7,8,9,10,11,12,13])
        fe_cols = ['rr_w13', 'rr_w12', 'rr_w11', 'rr_w10', 'rr_w09','rr_w08', 'rr_w07', 
                   'rr_w06','rr_w05', 'rr_w04', 'rr_w03', 'rr_w02','rr_w01']  
        activedays_cols = ['active_rr_days_w13', 'active_rr_days_w12',
           'active_rr_days_w11', 'active_rr_days_w10', 'active_rr_days_w09',
           'active_rr_days_w08', 'active_rr_days_w07', 'active_rr_days_w06',
           'active_rr_days_w05', 'active_rr_days_w04', 'active_rr_days_w03',
           'active_rr_days_w02', 'active_rr_days_w01'] 
        norm_intent=['rr_norm_w13', 'rr_norm_w12', 'rr_norm_w11', 'rr_norm_w10',
           'rr_norm_w09', 'rr_norm_w08', 'rr_norm_w07', 'rr_norm_w06',
           'rr_norm_w05', 'rr_norm_w04', 'rr_norm_w03', 'rr_norm_w02',
           'rr_norm_w01']
        fe_data['fe2rr_91days'] = fe_data.apply(lambda x: (x['rr_91days']/x['fe_91days']) if x['fe_91days'] > 0 else 0.5,axis=1)
        
        print('cp1')
        fe_data['rr_intent_multiplier'] = fe_data['rapido_age'].apply(lambda x: correction_factor(rr_weight,x)[0])
        fe_data['rr_need'] = np.array(fe_data[fe_cols]).max(axis=1)
        fe_data['no_week_rr_done'] =  back_up[fe_cols].count(axis=1)
        fe_data['%week_rr_done'] = fe_data.apply(lambda x: x['no_week_rr_done']*(1/(13 - correction_factor(rr_weight,x['rapido_age'])[1])), axis = 1 )
#         fe_data['rr_cov'] =  (np.sqrt(back_up[fe_cols].var(axis=1))/back_up[fe_cols].mean(axis=1))
#         fe_data['rr_volatility_type'] = fe_data.apply(lambda x: 'Low' if x['rr_cov'] <= 0.4 else 'High', axis = 1)
        fe_data['rr_mean_intent'] =  back_up[fe_cols].mean(axis=1)
        fe_data['rr_mean_intent_norm'] =  back_up[norm_intent].mean(axis=1)
        fe_data['rr_mean_intent_type'] = fe_data['rr_mean_intent'].apply(lambda x: 'Low' if x <= 3 else 'High')
        fe_data['rr_mean_intent_final'] = fe_data['rr_mean_intent'].apply(lambda x: intent_decider(x))
#         fe_data['rr_potential'] = fe_data['rr_need'] - fe_data['rr_mean_intent']
#         fe_data['rr_potential_per'] = (fe_data['rr_need'] - fe_data['rr_mean_intent'])/fe_data['rr_mean_intent']
#         fe_data['rr_potential_per_type'] = fe_data['rr_potential_per'].apply(lambda x: 'High' if x >= 0.5 else 'Low')
        fe_data['rr_trend_n'] = (  ((13 - fe_data.apply(lambda x: correction_factor(rr_weight,x['rapido_age'])[1],axis=1)) * (np.array(fe_data[fe_cols])*rr_weight).sum(axis=1))  - (np.array(fe_data[fe_cols]).sum(axis=1) * fe_data.apply(lambda x: correction_factor(rr_weight,x['rapido_age'])[2],axis=1))  ) 
        fe_data['rr_trend_d'] = ( ((13 - fe_data.apply(lambda x: correction_factor(rr_weight,x['rapido_age'])[1],axis = 1)) * fe_data.apply(lambda x: correction_factor(rr_weight,x['rapido_age'])[3],axis =1)) -(fe_data.apply(lambda x: correction_factor(rr_weight,x['rapido_age'])[2],axis =1))**2 )
        fe_data['rr_trend'] = fe_data['rr_trend_n']/fe_data['rr_trend_d']
        fe_data['rr_trend'] = fe_data['rr_trend'].fillna(0)
        fe_data['rr_intent_trend_type'] = fe_data['rr_trend'].apply(lambda x: intent_trend_type(x))
        fe=fe_data[fe_cols]
        fe=fe.ge(1.0).astype(int)
        fe_data['rr_regularity']= (1/91)*(np.array(fe)*rr_weight).sum(axis=1)
        fe_data['rr_regularity_scaled'] = fe_data['rr_regularity']*fe_data['rr_intent_multiplier']
        fe_data['rr_regularity_type'] = fe_data['rr_regularity_scaled'].apply(lambda x: 'Low' if x <= 0.3 else 'High')
        fe_data['rr_recency_type'] = fe_data['days_since_last_rr'].apply(lambda x: recency_type(x))
        fe_data['active_days_mean'] =  back_up[activedays_cols].mean(axis=1)
        fe_data['rr_regularity_final_old'] = fe_data.apply(lambda x: regularity_decider(x['rr_regularity_scaled'],x['active_days_mean']),axis=1)
        print('cp2')
        fe_data['rr_regularity_final'] = fe_data.apply(lambda x: regularity_decider_v2(x['rr_regularity_scaled'],x['active_days_mean']),axis=1)
        fe_data['reg_intent']=fe_data.rr_regularity_final+'_'+fe_data.rr_mean_intent_final
        print('cp3')
#         fe_data['consideration_final'] = fe_data.apply(lambda x: consideration_signal_creator(x['rr_regularity_scaled'],x['rr_mean_intent'],x['active_days_mean']),axis=1)
#         fe_data['rr_intent_trend_type_v2']= fe_data.apply(lambda x: custom_func(x['rr_mean_intent_type'],x['rr_intent_trend_type']),axis=1)

        return(fe_data)

    

    print('here')
    L3_signals = create_L3_signals(fe_data,back_up)
            
            

    L3_signals['city']=city_name
    Final_data=L3_signals.copy()
    
#     Final_data = Final_data.add_suffix('_nov22')
    # Final_data['target']=np.where(Final_data.next_lts=='DORMANT',1,0)
    return Final_data

    

In [62]:
date_list=['2023-11-04','2023-11-11','2023-11-18','2023-11-25','2023-12-02','2023-12-09','2023-12-16','2023-12-23']

# date_list=['2023-11-04','2023-11-11','2023-11-18']


In [59]:
base_week=date_list[0]
city='Bangalore'

In [63]:
concatenated_df=pd.DataFrame()
for i in date_list:
    print(i)
    tdf=consideration_query(i,city,base_week)
    print(tdf.shape[0])
    concatenated_df=concatenated_df.append(tdf).reset_index(drop=True)



2023-11-04
(460980, 93)
here
cp1
cp2
cp3
448748
2023-11-11
(460980, 93)
here
cp1
cp2
cp3
453039
2023-11-18
(460980, 93)
here
cp1
cp2
cp3
452858
2023-11-25
(460980, 93)
here
cp1
cp2
cp3
452762
2023-12-02
(460980, 93)
here
cp1
cp2
cp3
452780
2023-12-09
(460980, 93)
here
cp1
cp2
cp3
452632
2023-12-16
(460980, 93)
here
cp1
cp2
cp3
452442
2023-12-23
(460980, 93)
here
cp1
cp2
cp3
452216


In [65]:
concatenated_df['reg_intent']=concatenated_df.rr_regularity_final+'_'+concatenated_df.rr_mean_intent_final
base_df=concatenated_df[concatenated_df.run_date==base_week].reset_index(drop=True)

In [66]:
def stability_func(df,base_week):
    df_base_week=df[df.run_date==base_week][['customer','run_date','rr_regularity_final','rr_mean_intent_final','reg_intent']].reset_index(drop=True)
    # ranking = {'rarely_1_rr':1,
    #            'rarely_2+_rr':2,
    #            'once_in_4_weeks_1_rr':3,
    #            'once_in_4_weeks_2+_rr':4,
    #         'once_in_2_weeks_1_rr':5,
    #            'once_in_2_weeks_2+_rr':6,
    #            'weekly_1_rr':7,
    #            'weekly_2+_rr':8}

    ranking = {
               '05_rare_need':1,
            '04_monthly':2,
               '03_bi_weekly':3,
               '02_weekly':4,
               '01_daily':5}

    intent_ranking = {
               '03_Low':1,
            '02_Medium':2,
               '01_High':3}


    final_rank={
    '05_rare_need_03_Low' : 1,
    '05_rare_need_02_Medium' : 2,
    '05_rare_need_01_High' : 3,
    '04_monthly_03_Low' : 4,
    '04_monthly_02_Medium' : 5,
    '04_monthly_01_High' : 6,
    '03_bi_weekly_03_Low' : 7,
    '03_bi_weekly_02_Medium' : 8,
    '03_bi_weekly_01_High' : 9,
    '02_weekly_03_Low' : 10,
    '02_weekly_02_Medium' : 11,
    '02_weekly_01_High' : 12,
    '01_daily_02_Medium' : 13,
    '01_daily_01_High' : 14
    }    

    df['reg_rank'] = df['rr_regularity_final'].map(ranking)
    df['int_rank'] = df['rr_mean_intent_final'].map(intent_ranking)
    df['final_rank'] = df['reg_intent'].map(final_rank)
    
    ## Regularity Stabillity

    pvt_df=pd.pivot_table(df, values='reg_rank', index='customer', columns='run_date', aggfunc='sum').reset_index()


    pvt_df.columns


    dropped_reg = pvt_df.iloc[:, 1:].lt(pvt_df[base_week], axis=0)

    # Display the result
    print(dropped_reg.head())




    dropped_reg.head()

    first_dropped_week = dropped_reg.idxmax(axis=1)


    pvt_df['first_dropped_week'] = first_dropped_week.where(dropped_reg.any(axis=1), 'Maintained/Upgraded')




    pvt_df.reset_index(inplace=True)

    pvt_df.head()

    df_base_week_stability=pvt_df.merge(df_base_week,on=['customer'],how='inner')

    reg_stab=df_base_week_stability.groupby(by=['run_date','rr_regularity_final','first_dropped_week'],as_index=False).agg(
    user_count=('customer','nunique')).pivot_table( values='user_count', index=['run_date','rr_regularity_final',], columns='first_dropped_week', aggfunc='sum').reset_index().merge(df_base_week.groupby(by=['rr_regularity_final'],as_index=False).agg(
    base=('customer','nunique')),on=['rr_regularity_final'],how='inner').fillna(0)
#     .to_csv(f'{month}/output_files/regularity_stability.csv',index=False)

    ## Intent Stabillity

    pvt_df_int=pd.pivot_table(df, values='int_rank', index='customer', columns='run_date', aggfunc='sum').reset_index()


    pvt_df_int.columns


    dropped_int = pvt_df_int.iloc[:, 1:].lt(pvt_df_int[base_week], axis=0)

    # Display the result
    print(dropped_int.head())




    dropped_int.head()

    int_first_dropped_week = dropped_int.idxmax(axis=1)


    pvt_df_int['int_first_dropped_week'] = int_first_dropped_week.where(dropped_int.any(axis=1), 'Maintained/Upgraded')




    pvt_df_int.reset_index(inplace=True)

    pvt_df_int.head()

    df_base_week_int_stability=pvt_df_int.merge(df_base_week,on=['customer'],how='inner')

    int_stab=df_base_week_int_stability.groupby(by=['run_date','rr_mean_intent_final','int_first_dropped_week'],as_index=False).agg(
    user_count=('customer','nunique')).pivot_table( values='user_count', index=['run_date','rr_mean_intent_final',], columns='int_first_dropped_week', aggfunc='sum').reset_index().merge(df_base_week.groupby(by=['rr_mean_intent_final'],as_index=False).agg(base=('customer','nunique')),on=['rr_mean_intent_final'],how='inner').fillna(0)
#     .to_csv(f'{month}/output_files/intent_stability.csv',index=False)

    ## Final Stabillity

    pvt_df_final=pd.pivot_table(df, values='final_rank', index='customer', columns='run_date', aggfunc='sum').reset_index()


    pvt_df_final.columns


    dropped_final = pvt_df_final.iloc[:, 1:].lt(pvt_df_final[base_week], axis=0)

    # Display the result
    print(dropped_final.head())




    dropped_final.head()

    final_first_dropped_week = dropped_final.idxmax(axis=1)


    pvt_df_final['final_first_dropped_week'] = final_first_dropped_week.where(dropped_final.any(axis=1), 'Maintained/Upgraded')




    pvt_df_final.reset_index(inplace=True)

    pvt_df_final.head()

    df_base_week_final_stability=pvt_df_final.merge(df_base_week,on=['customer'],how='inner')

    final_stab=df_base_week_final_stability.groupby(by=['run_date','rr_regularity_final','rr_mean_intent_final','final_first_dropped_week'],as_index=False).agg(
    user_count=('customer','nunique')).pivot_table( values='user_count', index=['run_date','rr_regularity_final','rr_mean_intent_final'], columns='final_first_dropped_week', aggfunc='sum').reset_index().merge(df_base_week.groupby(by=['rr_regularity_final','rr_mean_intent_final'],as_index=False).agg(base=('customer','nunique')),on=['rr_regularity_final','rr_mean_intent_final'],how='inner').fillna(0)
#     .to_csv(f'{month}/output_files/final_stability.csv',index=False)

    return reg_stab,int_stab,final_stab



In [67]:
reg_df,int_df,final_seg_df=stability_func(concatenated_df,base_week)

run_date  2023-11-04  2023-11-11  2023-11-18  2023-11-25  2023-12-02  \
0              False       False       False       False       False   
1              False        True       False        True        True   
2              False       False       False       False       False   
3              False       False       False       False       False   
4              False       False        True        True        True   

run_date  2023-12-09  2023-12-16  2023-12-23  
0              False       False       False  
1              False       False       False  
2              False        True        True  
3              False       False       False  
4              False       False       False  
run_date  2023-11-04  2023-11-11  2023-11-18  2023-11-25  2023-12-02  \
0              False       False       False       False       False   
1              False       False       False       False       False   
2              False       False       False       False       False 

In [115]:
# complete flow
def stab_func_2(df,base_week,base_df,cons_type):
    
#     base_df=df[df.run_date==base_week].reset_index(drop=True)
    fdf1=base_df.groupby(by=['city','run_date',cons_type],as_index=False).agg(base=('customer','nunique'))
    j=0
    for i in date_list:
        j+=1
        if i!=base_week:
            tdf=df[df.run_date==base_week][['run_date','customer',cons_type]].merge(df[df.run_date==i][['customer',cons_type]],on=['customer'],how='left').pivot_table(values='customer', index=['run_date',cons_type+'_x'], columns=cons_type+'_y', aggfunc='nunique').reset_index().fillna(0)
#             print(list(tdf))
            items_to_remove=['run_date',cons_type+'_x']
            filtered_list = [x for x in list(tdf) if x not in items_to_remove]
            tdf.columns=['run_date',cons_type]+[j+'_'+i for j in filtered_list]
            fdf1=fdf1.merge(tdf,on=['run_date',cons_type],how='inner')
    return fdf1
        
        

In [116]:
#cons_type: 'rr_regularity_final','rr_mean_intent_final','reg_intent'
finaldf=stab_func_2(concatenated_df,base_week,base_df,'reg_intent')

In [118]:
finaldf

,city,run_date,reg_intent,base,01_daily_01_High_2023-11-11,01_daily_02_Medium_2023-11-11,02_weekly_01_High_2023-11-11,02_weekly_02_Medium_2023-11-11,02_weekly_03_Low_2023-11-11,03_bi_weekly_01_High_2023-11-11,03_bi_weekly_02_Medium_2023-11-11,03_bi_weekly_03_Low_2023-11-11,04_monthly_01_High_2023-11-11,04_monthly_02_Medium_2023-11-11,04_monthly_03_Low_2023-11-11,05_rare_need_01_High_2023-11-11,05_rare_need_02_Medium_2023-11-11,05_rare_need_03_Low_2023-11-11,01_daily_01_High_2023-11-18,01_daily_02_Medium_2023-11-18,02_weekly_01_High_2023-11-18,02_weekly_02_Medium_2023-11-18,02_weekly_03_Low_2023-11-18,03_bi_weekly_01_High_2023-11-18,03_bi_weekly_02_Medium_2023-11-18,03_bi_weekly_03_Low_2023-11-18,04_monthly_01_High_2023-11-18,04_monthly_02_Medium_2023-11-18,04_monthly_03_Low_2023-11-18,05_rare_need_01_High_2023-11-18,05_rare_need_02_Medium_2023-11-18,05_rare_need_03_Low_2023-11-18,01_daily_01_High_2023-11-25,01_daily_02_Medium_2023-11-25,02_weekly_01_High_2023-11-25,02_weekly_02_Medium_2023-11-25,02_weekly_03_Low_2023-11-25,03_bi_weekly_01_High_2023-11-25,03_bi_weekly_02_Medium_2023-11-25,03_bi_weekly_03_Low_2023-11-25,04_monthly_01_High_2023-11-25,04_monthly_02_Medium_2023-11-25,04_monthly_03_Low_2023-11-25,05_rare_need_01_High_2023-11-25,05_rare_need_02_Medium_2023-11-25,05_rare_need_03_Low_2023-11-25,01_daily_01_High_2023-12-02,01_daily_02_Medium_2023-12-02,02_weekly_01_High_2023-12-02,02_weekly_02_Medium_2023-12-02,02_weekly_03_Low_2023-12-02,03_bi_weekly_01_High_2023-12-02,03_bi_weekly_02_Medium_2023-12-02,03_bi_weekly_03_Low_2023-12-02,04_monthly_01_High_2023-12-02,04_monthly_02_Medium_2023-12-02,04_monthly_03_Low_2023-12-02,05_rare_need_01_High_2023-12-02,05_rare_need_02_Medium_2023-12-02,05_rare_need_03_Low_2023-12-02,01_daily_01_High_2023-12-09,01_daily_02_Medium_2023-12-09,02_weekly_01_High_2023-12-09,02_weekly_02_Medium_2023-12-09,02_weekly_03_Low_2023-12-09,03_bi_weekly_01_High_2023-12-09,03_bi_weekly_02_Medium_2023-12-09,03_bi_weekly_03_Low_2023-12-09,04_monthly_01_High_2023-12-09,04_monthly_02_Medium_2023-12-09,04_monthly_03_Low_2023-12-09,05_rare_need_01_High_2023-12-09,05_rare_need_02_Medium_2023-12-09,05_rare_need_03_Low_2023-12-09,01_daily_01_High_2023-12-16,01_daily_02_Medium_2023-12-16,02_weekly_01_High_2023-12-16,02_weekly_02_Medium_2023-12-16,02_weekly_03_Low_2023-12-16,03_bi_weekly_01_High_2023-12-16,03_bi_weekly_02_Medium_2023-12-16,03_bi_weekly_03_Low_2023-12-16,04_monthly_01_High_2023-12-16,04_monthly_02_Medium_2023-12-16,04_monthly_03_Low_2023-12-16,05_rare_need_01_High_2023-12-16,05_rare_need_02_Medium_2023-12-16,05_rare_need_03_Low_2023-12-16,01_daily_01_High_2023-12-23,01_daily_02_Medium_2023-12-23,02_weekly_01_High_2023-12-23,02_weekly_02_Medium_2023-12-23,02_weekly_03_Low_2023-12-23,03_bi_weekly_01_High_2023-12-23,03_bi_weekly_02_Medium_2023-12-23,03_bi_weekly_03_Low_2023-12-23,04_monthly_01_High_2023-12-23,04_monthly_02_Medium_2023-12-23,04_monthly_03_Low_2023-12-23,05_rare_need_01_High_2023-12-23,05_rare_need_02_Medium_2023-12-23,05_rare_need_03_Low_2023-12-23
0,Bangalore,2023-11-04,01_daily_01_High,52496,48554.0,1917.0,545.0,505.0,0.0,940.0,35.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,45504.0,2627.0,659.0,908.0,0.0,2234.0,352.0,0.0,212.0,0.0,0.0,0.0,0.0,0.0,42214.0,3061.0,683.0,1378.0,0.0,4121.0,726.0,3.0,308.0,2.0,0.0,0.0,0.0,0.0,40250.0,2959.0,589.0,1452.0,1.0,5490.0,1015.0,4.0,457.0,107.0,0.0,172.0,0.0,0.0,37702.0,3466.0,549.0,1988.0,12.0,5313.0,1514.0,5.0,1507.0,275.0,3.0,162.0,0.0,0.0,35513.0,3979.0,566.0,2501.0,14.0,4713.0,2047.0,21.0,2288.0,578.0,5.0,261.0,10.0,0.0,33651.0,4116.0,536.0,2905.0,19.0,4635.0,2600.0,47.0,2560.0,805.0,16.0,480.0,124.0,2.0
1,Bangalore,2023-11-04,01_daily_02_Medium,22209,2413.0,16553.0,10.0,2534.0,9.0,15.0,675.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2781.0,14139.0,13.0,3312.0,8.0,118.0,1723.0,11.0,0.0,104.0,0.0,0.0,0.0,0.0,3415.0,11943.0,20.0,3733.0,22.0,256.0,2662.0,14.0,2.0,142.0,0.0,0.0,0.0,0.0,3961.0,10599.0,17.0,3489.0,19.0,409.0,3262.0,19.0,26.0,318.0,10.0,0.0,80.0,0.0,4125.0,

In [120]:
base_df.groupby(by=['city_name','run_date','reg_intent'],as_index=False).agg(
    user_count=('customer','nunique'),
    fe_retention_w01=('fe_flag_w01','sum'),
    fe_retention_w02=('fe_flag_w02','sum'),
    fe_retention_w03=('fe_flag_w03','sum'),
    fe_retention_w04=('fe_flag_w04','sum'),
    fe_retention_w05=('fe_flag_w05','sum'),
    fe_retention_w06=('fe_flag_w06','sum'),
    fe_retention_w07=('fe_flag_w07','sum'),
    fe_retention_w08=('fe_flag_w08','sum'),
    fe_retention_w09=('fe_flag_w09','sum'),
    fe_retention_w10=('fe_flag_w10','sum'),
    
    rr_retention_w01=('rr_flag_w01','sum'),
    rr_retention_w02=('rr_flag_w02','sum'),
    rr_retention_w03=('rr_flag_w03','sum'),
    rr_retention_w04=('rr_flag_w04','sum'),
    rr_retention_w05=('rr_flag_w05','sum'),
    rr_retention_w06=('rr_flag_w06','sum'),
    rr_retention_w07=('rr_flag_w07','sum'),
    rr_retention_w08=('rr_flag_w08','sum'),
    rr_retention_w09=('rr_flag_w09','sum'),
    rr_retention_w10=('rr_flag_w10','sum'),
        
    nr_retention_w01=('nr_flag_w01','sum'),
    nr_retention_w02=('nr_flag_w02','sum'),
    nr_retention_w03=('nr_flag_w03','sum'),
    nr_retention_w04=('nr_flag_w04','sum'),
    nr_retention_w05=('nr_flag_w05','sum'),
    nr_retention_w06=('nr_flag_w06','sum'),
    nr_retention_w07=('nr_flag_w07','sum'),
    nr_retention_w08=('nr_flag_w08','sum'),
    nr_retention_w09=('nr_flag_w09','sum'),
    nr_retention_w10=('nr_flag_w10','sum')
    
    )
# .to_csv(f'{month}/output_files/reg_retention_10w_newlogic.csv',index=False)

#rr_mean_intent_final, rr_regularity_final, reg_intent

# gender_tag,
# taxi_income_segment,
# ps_tag_link,
# ps_tag_auto,
# taxi_service_affinity,
# taxi_distance_preference,
# taxi_time_of_day_preference,
# taxi_day_of_week_preference

,city_name,run_date,reg_intent,user_count,fe_retention_w01,fe_retention_w02,fe_retention_w03,fe_retention_w04,fe_retention_w05,fe_retention_w06,fe_retention_w07,fe_retention_w08,fe_retention_w09,fe_retention_w10,rr_retention_w01,rr_retention_w02,rr_retention_w03,rr_retention_w04,rr_retention_w05,rr_retention_w06,rr_retention_w07,rr_retention_w08,rr_retention_w09,rr_retention_w10,nr_retention_w01,nr_retention_w02,nr_retention_w03,nr_retention_w04,nr_retention_w05,nr_retention_w06,nr_retention_w07,nr_retention_w08,nr_retention_w09,nr_retention_w10
0,Bangalore,2023-11-04,01_daily_01_High,52496,43886,45898,48626,50158,50696,51000,51175,51258,51343,51448,46237,48105,49666,50387,50780,51025,51168,51243,51312,51406,43612,46494,48555,49604,50177,50562,50766,50879,50976,51108
1,Bangalore,2023-11-04,01_daily_02_Medium,22209,18515,19519,20530,21172,21421,21563,21631,21679,21712,21756,18906,20007,20753,21136,21364,21487,21560,21603,21639,21681,16964,18748,19899,20503,20859,21093,21208,21270,21350,21432
2,Bangalore,2023-11-04,02_weekly_01_High,3642,2694,2944,3154,3267,3342,3370,3384,3399,3411,3421,2732,3000,3182,3269,3325,3350,3368,3385,3392,3402,2465,2822,3052,3172,3250,3291,3320,3338,3353,3368
3,Bangalore,2023-11-04,02_weekly_02_Medium,82989,62995,70074,74542,77250,78641,79423,79873,80119,80362,80569,61508,69706,74067,76379,77800,78707,79243,79526,79822,80088,52030,62827,68843,72363,74587,76078,77042,77537,78042,78502
4,Bangalore,2023-11-04,02_weekly_03_Low,23077,14590,17189,18677,19570,20079,20394,20607,20745,20845,20955,13301,16183,17780,18713,19306,19724,19992,20164,20298,20456,10591,13884,15793,17078,17904,18506,18919,19155,19361,19576
5,Bangalore,2023-11-04,03_bi_weekly_01_High,5874,4277,4726,5075,5283,5400,5473,5516,5543,5569,5596,4315,4780,5074,5241,5350,5426,5462,5490,5523,5553,3936,4495,4857,5076,5220,5317,5370,5401,5440,5474
6,Bangalore,2023-11-04,03_bi_weekly_02_Medium,65393,43151,50883,55602,58399,60086,61174,61817,62242,62591,62887,40157,48612,53638,56518,58409,59706,60539,61063,61563,61950,32751,42271,48205,51983,54539,56368,57621,58440,59191,59801
7,Bangalore,2023-11-04,03_bi_weekly_03_Low,64042,36148,45404,51273,54837,57061,58520,59485,60085,60606,61004,31405,40863,47267,51178,53830,55656,56910,57746,58470,59062,23850,33496,40282,44899,48157,50589,52418,53654,54743,55641
8,Bangalore,2023-11-04,04_monthly_01_High,2401,1486,1645,1791,1880,1953,2006,2033,2056,2083,2103,1458,1634,1762,1842,1912,1965,1992,2010,2040,2062,1380,1588,1737,1825,1898,1948,1983,2006,2035,2058
9,Bangalore,2023-11-04,04_monthly_02_Medium,25627,13429,16713,18857,20256,21286,21902,22343,22668,22930,23165,11886,15162,17337,18787,19910,20653,21180,21576,21923,22221,9699,13003,15229,16870,18128,19058,19731,20278,20726,21104


In [103]:
test=list(finaldf)

In [108]:
test.remove([['city','01_daily_2023-11-11']])

ValueError: list.remove(x): x not in list

In [85]:
test=['city']+[i+'_yo' for i in test]

In [105]:
test

['city',
 'run_date',
 'rr_regularity_final',
 'base',
 '01_daily_2023-11-11',
 '02_weekly_2023-11-11',
 '03_bi_weekly_2023-11-11',
 '04_monthly_2023-11-11',
 '05_rare_need_2023-11-11',
 '01_daily_2023-11-18',
 '02_weekly_2023-11-18',
 '03_bi_weekly_2023-11-18',
 '04_monthly_2023-11-18',
 '05_rare_need_2023-11-18',
 '01_daily_2023-11-25',
 '02_weekly_2023-11-25',
 '03_bi_weekly_2023-11-25',
 '04_monthly_2023-11-25',
 '05_rare_need_2023-11-25',
 '01_daily_2023-12-02',
 '02_weekly_2023-12-02',
 '03_bi_weekly_2023-12-02',
 '04_monthly_2023-12-02',
 '05_rare_need_2023-12-02',
 '01_daily_2023-12-09',
 '02_weekly_2023-12-09',
 '03_bi_weekly_2023-12-09',
 '04_monthly_2023-12-09',
 '05_rare_need_2023-12-09',
 '01_daily_2023-12-16',
 '02_weekly_2023-12-16',
 '03_bi_weekly_2023-12-16',
 '04_monthly_2023-12-16',
 '05_rare_need_2023-12-16',
 '01_daily_2023-12-23',
 '02_weekly_2023-12-23',
 '03_bi_weekly_2023-12-23',
 '04_monthly_2023-12-23',
 '05_rare_need_2023-12-23']